# Movie Revenue Prediction ML Pipeline
This notebook demonstrates an end-to-end Machine Learning pipeline to predict movie box office revenue using the TMDB dataset and a Random Forest Regressor.

## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import ast
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Set aesthetic parameters
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

## 2. Load Dataset

In [ ]:
file_path = 'dataset_1_collected_data.csv'
df = pd.read_csv(file_path)

print("Head of the dataset:")
display(df.head())

print("\nDataset Info:")
df.info()

print("\nStatistical Summary:")
display(df.describe())

## 3. Data Cleaning
- Handle missing values
- Replace budget=0 with median
- Convert release_date into release_year and release_month

In [ ]:
# Drop unnecessary columns
cols_to_drop = ['id', 'title', 'production_companies', 'production_countries', 'cast', 'director']
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

# Handle missing values for numeric columns
numeric_cols = df.select_dtypes(include=[np.number]).columns
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

# Replace budget = 0 with median of non-zero budgets
median_budget = df[df['budget'] > 0]['budget'].median()
df['budget'] = df['budget'].replace(0, median_budget)

# Convert release_date and extract features
df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')
df = df.dropna(subset=['release_date']) 
df['release_year'] = df['release_date'].dt.year
df['release_month'] = df['release_date'].dt.month
df = df.drop(columns=['release_date'])

print("Data cleaning complete.")

## 4. Feature Engineering
- Parse genres
- One-hot encoding for genres and language
- Log transformation for financial columns

In [ ]:
# Function to parse stringified lists of genres
def parse_genres(genre_str):
    try:
        genres = ast.literal_eval(genre_str)
        return genres if isinstance(genres, list) else []
    except:
        return []

if 'genres' in df.columns:
    df['genres_list'] = df['genres'].apply(parse_genres)
    all_genres = [genre for sublist in df['genres_list'] for genre in sublist]
    top_genres = pd.Series(all_genres).value_counts().head(10).index.tolist()
    for genre in top_genres:
        df[f'genre_{genre}'] = df['genres_list'].apply(lambda x: 1 if genre in x else 0)
    df = df.drop(columns=['genres', 'genres_list'])

# Encode original_language (Top 5)
if 'original_language' in df.columns:
    top_langs = df['original_language'].value_counts().head(5).index.tolist()
    df['original_language'] = df['original_language'].apply(lambda x: x if x in top_langs else 'other')
    df = pd.get_dummies(df, columns=['original_language'], prefix='lang')

# Apply Log Transformation to Budget and Revenue
df['log_budget'] = np.log1p(df['budget'])
df['log_revenue'] = np.log1p(df['revenue'])

print("Feature engineering complete.")

## 5. Exploratory Data Analysis (EDA)

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(12, 10))
numeric_df = df.select_dtypes(include=[np.number])
sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Heatmap')
plt.show()

# Budget vs Revenue
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='budget', y='revenue', alpha=0.5)
plt.title('Budget vs Revenue')
plt.show()

# Popularity vs Revenue
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='popularity', y='revenue', alpha=0.5, color='orange')
plt.title('Popularity vs Revenue')
plt.show()

# Revenue Distribution
plt.figure(figsize=(10, 6))
sns.histplot(df['revenue'], bins=50, kde=True)
plt.title('Revenue Distribution')
plt.show()

## 6. Data Preprocessing

In [ ]:
# Define features (X) and target (y)
X = df.drop(columns=['revenue', 'log_revenue', 'budget'])
y = df['log_revenue']

# Train-Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

## 7 & 8. Model Training and Evaluation

In [ ]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)

# Evaluation
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)

print(f"Baseline R2 Score: {r2:.4f}")
print(f"Baseline RMSE: {rmse:.4f}")
print(f"Baseline MAE: {mae:.4f}")

## 9. Hyperparameter Tuning

In [ ]:
param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5]
}

grid_search = GridSearchCV(RandomForestRegressor(random_state=42), param_grid, cv=3, scoring='r2', n_jobs=-1)
grid_search.fit(X_train, y_train)

print("Best Parameters:", grid_search.best_params_)
best_rf = grid_search.best_estimator_

## 10. Model Performance Table

In [ ]:
y_pred_tuned = best_rf.predict(X_test)
metrics = {
    'R2 Score': r2_score(y_test, y_pred_tuned),
    'RMSE': np.sqrt(mean_squared_error(y_test, y_pred_tuned)),
    'MAE': mean_absolute_error(y_test, y_pred_tuned)
}

performance_df = pd.DataFrame(list(metrics.items()), columns=['Metric', 'Value'])
display(performance_df)

## 11. Feature Importance

In [ ]:
importances = best_rf.feature_importances_
indices = np.argsort(importances)[::-1]
feature_names = X.columns

plt.figure(figsize=(12, 8))
sns.barplot(x=importances[indices][:15], y=feature_names[indices][:15])
plt.title('Top 15 Feature Importances')
plt.show()

## 12 & 13. Final Prediction and Visualization

In [ ]:
plt.figure(figsize=(10, 10))
plt.scatter(y_test, y_pred_tuned, alpha=0.3)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], '--r', linewidth=2)
plt.xlabel('Actual (Log Revenue)')
plt.ylabel('Predicted (Log Revenue)')
plt.title('Actual vs Predicted Revenue (Log Scale)')
plt.show()

## 14. Save the Model

In [ ]:
joblib.dump(best_rf, 'movie_revenue_model.pkl')
print("Model saved as 'movie_revenue_model.pkl'")